# 03 · Corridor Exposure & Vulnerable Road User (VRU) Synthesis

This notebook cross-links the national bicycle collision typology findings (`2010–2021` microdata and `2006–2021` `M2_full_typology` severity model) with Phase 1 Toronto Centreline + AADT exposure-normalized corridor rates (`2006–2023`):

1. **Functional Road Class Exposure vs. Spatial Density (`toronto_cyclist_corridor_exposure_by_class.csv`)**: Compares `cyclist_ksi_per_100m_vkt` (`0.751` on Minor Arterials vs `0.519` on Major Arterials) with spatial linear density `cyclist_ksi_per_km` (`0.674` on Major Arterials vs `0.485` on Minor Arterials), showing that Major and Minor Arterials concentrate `82.32%` (`447 / 543`) of AADT-monitored cyclist KSI crashes.
2. **High-Risk Toronto Cycling Corridors (`toronto_top_cyclist_ksi_corridors.csv`)**: Evaluates top corridors by count and VKT rate (`Bloor St W`, `Spadina Ave`, `Danforth Ave`, `Yonge St`, `St Clair Ave W`, `Davenport Rd`, `Bay St`, `Dupont St`, `Dundas St E`) alongside network-density reference corridors (`College St`, `Harbord St`).
3. **National `M2_full_typology` Bicyclist Severity Discrimination Lift**: Evaluates out-of-time `ROC-AUC` improvement (`0.6901 -> 0.7852` on `2017–2019` validation; `0.6810 -> 0.7576` on `2020–2021` test) and `heavy_freight_x_vru` odds ratios (`OR = 2.5716`).

In [1]:
import csv
from pathlib import Path
from bicycle_collisions.corridor_exposure_link import run_corridor_exposure_link

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src/bicycle_collisions").exists():
            return candidate
    raise RuntimeError("Could not locate projects/bicycles root")

ROOT = find_project_root()
synthesis_summary = run_corridor_exposure_link(ROOT)
class_rows = list(csv.DictReader((ROOT / "reports/tables/toronto_cyclist_corridor_exposure_by_class.csv").open()))

print("Toronto Cyclist KSI Exposure by Road Class (2006-2023):")
print("Road Class           | Segs  | Mon km   | Mean AADT | 100M VKT | Cyc KSI | Cyc/100M VKT | Cyc/km | Net Cyc KSI")
print("-" * 108)
for r in class_rows[:6]:
    print(
        f"{r['road_class']:20s} | {int(r['monitored_segments']):5,d} | {float(r['monitored_length_km']):8.3f} | "
        f"{float(r['mean_aadt']):9,.1f} | {float(r['cumulative_100m_vkt_18y']):8.4f} | {int(r['cyclist_ksi_crashes']):7d} | "
        f"{float(r['cyclist_ksi_per_100m_vkt']):12.3f} | {float(r['cyclist_ksi_per_km']):6.3f} | {int(r['network_cyclist_ksi_crashes']):11d}"
    )

Toronto Cyclist KSI Exposure by Road Class (2006-2023):
Road Class           | Segs  | Mon km   | Mean AADT | 100M VKT | Cyc KSI | Cyc/100M VKT | Cyc/km | Net Cyc KSI
------------------------------------------------------------------------------------------------------------
Major Arterial       | 3,031 |  479.494 |  18,287.1 | 621.7977 |     323 |        0.519 |  0.674 |         493
Minor Arterial       | 1,696 |  255.591 |   9,574.2 | 165.2028 |     124 |        0.751 |  0.485 |         219
Collector            | 2,768 |  427.137 |   3,627.0 | 104.9402 |      54 |        0.515 |  0.126 |         100
Local                | 6,069 | 1015.278 |   1,085.9 |  70.0861 |      40 |        0.571 |  0.039 |         107
Expressway Ramp      |   259 |   77.967 |   9,656.7 |  47.5057 |       1 |        0.021 |  0.013 |           1
Expressway           |    38 |   12.063 |  29,660.2 |  26.0240 |       0 |        0.000 |  0.000 |           1


## 2. Top Toronto Corridors: VKT-Normalized Rates, Spatial Density & Freight Co-Exposure

We inspect `reports/tables/toronto_top_cyclist_ksi_corridors.csv`, examining key cycling corridors (`Bloor St W`, `Spadina Ave`, `Danforth Ave`, `Yonge St`, `St Clair Ave W`, `Davenport Rd`, `College St`, `Harbord St`, `Bay St`, `Dupont St`, `Dundas St E`) across both AADT-monitored segments (`cyclist_ksi_per_100m_vkt`) and total network length (`network_cyclist_ksi_per_km`).

In [2]:
corridors = list(csv.DictReader((ROOT / "reports/tables/toronto_top_cyclist_ksi_corridors.csv").open()))
print("Top Toronto Corridors (Monitored & Network Cyclist KSI Metrics):")
print("Corridor                             | Mon km | 100M VKT | Cyc KSI | Cyc/100M VKT | Cyc/km | Net Cyc | Net/km | HvyTrk KSI")
print("-" * 122)
for r in corridors:
    print(
        f"{r['corridor_name']:36s} | {float(r['monitored_length_km']):6.3f} | {float(r['cumulative_100m_vkt']):8.4f} | "
        f"{int(r['cyclist_ksi_crashes']):7d} | {float(r['cyclist_ksi_per_100m_vkt']):12.3f} | {float(r['cyclist_ksi_per_km']):6.3f} | "
        f"{int(r['network_cyclist_ksi_crashes']):7d} | {float(r['network_cyclist_ksi_per_km']):6.3f} | {int(r['heavy_truck_ksi_crashes']):10d}"
    )

Top Toronto Corridors (Monitored & Network Cyclist KSI Metrics):
Corridor                             | Mon km | 100M VKT | Cyc KSI | Cyc/100M VKT | Cyc/km | Net Cyc | Net/km | HvyTrk KSI
--------------------------------------------------------------------------------------------------------------------------
Bloor St W (Major Arterial)          | 11.375 |  10.7876 |      29 |        2.688 |  2.550 |      36 |  2.161 |          8
Yonge St (Major Arterial)            | 13.437 |  19.7873 |      21 |        1.061 |  1.563 |      26 |  1.477 |         25
Danforth Ave (Major Arterial)        |  5.059 |   5.6193 |      13 |        2.313 |  2.570 |      20 |  2.864 |          7
Spadina Ave (Major Arterial)         |  2.092 |   1.9986 |      12 |        6.004 |  5.736 |      14 |  4.513 |         20
Kingston Rd (Major Arterial)         | 14.451 |  19.9472 |      11 |        0.551 |  0.761 |      15 |  0.736 |          9
Avenue Rd (Major Arterial)           |  6.372 |   9.6844 |      10 |      

## 3. National `M2_full_typology` Bicyclist Severity Lift & Cross-Scale Synthesis

Finally, we verify the national `M2_full_typology` subgroup performance (`typology_subgroup_performance.csv`) and adjusted odds ratios (`typology_odds_ratios.csv`), confirming how explicit vehicle/VRU typology improves bicyclist fatality discrimination (`ROC-AUC 0.6810 -> 0.7576` on the `2020–2021` test split; `0.6901 -> 0.7852` on `2017–2019` validation).

In [3]:
rts_tables = ROOT.parent / "road_to_safety/reports/tables"
perf_rows = list(csv.DictReader((rts_tables / "typology_subgroup_performance.csv").open()))
or_rows = list(csv.DictReader((rts_tables / "typology_odds_ratios.csv").open()))

print("National Out-of-Time Severity Model Performance (Bicyclist & Heavy Freight x VRU Subgroups):")
for r in perf_rows:
    if r["subgroup"] in {"bicyclist", "heavy_freight_x_vru", "all_collisions"}:
        lift = float(r["m2_roc_auc"]) - float(r["m0_roc_auc"])
        print(
            f"  {r['split']:20s} | {r['subgroup']:20s} (n={int(r['n']):7,d}, fatal={int(r['fatal']):4,d}, rate={float(r['observed_fatality_rate_pct']):6.3f}%) | "
            f"ROC-AUC: M0={float(r['m0_roc_auc']):.4f} -> M2={float(r['m2_roc_auc']):.4f} ({lift:+.4f}) | "
            f"PR-AUC: M0={float(r['m0_pr_auc']):.4f} -> M2={float(r['m2_pr_auc']):.4f}"
        )

print("\nKey National Typology Adjusted Odds Ratios (M2_full_typology):")
for r in or_rows:
    if r["feature"] in {"has_road_tractor", "has_heavy_truck", "heavy_freight_x_vru", "has_light_duty", "has_bicyclist"}:
        print(
            f"  {r['feature']:22s}: OR={float(r['odds_ratio']):6.4f} | unadjusted_fatal_rate={float(r['unadjusted_fatality_rate_pct']):6.3f}% "
            f"({int(r['fatal_collisions_all_years']):,}/{int(r['collisions_all_years']):,})"
        )

National Out-of-Time Severity Model Performance (Bicyclist & Heavy Freight x VRU Subgroups):
  validation_2017_2019 | all_collisions       (n=330,376, fatal=5,072, rate= 1.535%) | ROC-AUC: M0=0.6945 -> M2=0.7692 (+0.0747) | PR-AUC: M0=0.0338 -> M2=0.0616
  validation_2017_2019 | bicyclist            (n= 14,940, fatal= 128, rate= 0.857%) | ROC-AUC: M0=0.6901 -> M2=0.7852 (+0.0951) | PR-AUC: M0=0.0308 -> M2=0.0648
  validation_2017_2019 | heavy_freight_x_vru  (n=  1,472, fatal= 193, rate=13.111%) | ROC-AUC: M0=0.6158 -> M2=0.6494 (+0.0336) | PR-AUC: M0=0.2339 -> M2=0.2652
  test_2020_2021       | all_collisions       (n=163,580, fatal=3,230, rate= 1.975%) | ROC-AUC: M0=0.6958 -> M2=0.7628 (+0.0670) | PR-AUC: M0=0.0415 -> M2=0.0747
  test_2020_2021       | bicyclist            (n=  8,885, fatal=  98, rate= 1.103%) | ROC-AUC: M0=0.6810 -> M2=0.7576 (+0.0766) | PR-AUC: M0=0.0359 -> M2=0.0475
  test_2020_2021       | heavy_freight_x_vru  (n=    853, fatal= 132, rate=15.475%) | ROC-AUC: M0=0.